                # SIMBA

                Sample trajectories, identify difficult examples, and add reflective rules or demonstrations in a sequence of improvement steps.

                **Use it when:** You want iterative, example-driven improvement and can tolerate a relatively expensive reflective search.

                **What compilation changes:** The selected program may add rules or demonstrations; this rerun's artifact shows which mechanism won.

                Important configuration in this benchmark:

                - six steps and four candidates in the full profile
- batch size capped at eight
- at most two demonstrations
- seed 42

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'simba'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='simba'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.SIMBA(
    metric=exact_match, bsize=8, num_candidates=profile.simba_candidates,
    max_steps=profile.simba_steps, max_demos=2, prompt_model=reflection_lm,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, seed=42)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

SIMBA — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 70.0%
uplift: +20.0 points vs Luna reference
optimization: $0.9482 and 18.8min
inference latency: mean 1.63s; p95 2.22s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/simba.json
- canonical prompt: chapter06/results/final_prompts/simba.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Compare the final training trajectory with the locked test score. A large gap is evidence that reflective search can still overfit a small benchmark.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (1288 characters):
Decide whether the supplied passage was generated by an AI.

When given short technical documentation or procedural instructions, do not classify it as AI merely because it uses numbered steps, concise wording, or a mechanical structure. Weigh source-like traits more heavily: exact domain API names, narrowly scoped instructions, inconsistent punctuation, awkward grammar, and lack of explanatory padding are compatible with human-authored documentation. For passages like this Spark procedure, treat the specific API references and lightly edited phrasing as evidence for `is_ai: false` unless stronger generation signals are present.

When the input is short, narrowly scoped technical or procedural documentation containing exact framework-specific identifiers—such as RDD, StructType, createDataFrame, and SparkSession—treat those source-like details as stronger evidence than numbered steps, repetitive wording, mechanical concis

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.